In [3]:
"This IPYNB file is only used as a test, and a tool for deploying the model. The real Code goes in the same file, but "
"with a .py extension rather than a .ipynb"

'with a .py extension rather than a .ipynb'

In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path("../../../../SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
    Path("backend/SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not locate usgs_data_USGS-01646500.csv")
print(f"Using CSV file at: {csv_path}")

df = pd.read_csv(csv_path)
df

Using CSV file at: ..\..\..\..\SOURCES_AND_DATASHEETS\usgs_data_USGS-01646500.csv


,Unnamed: 0,gage_height_ft,streamflow_cfs,dissolved_oxygen_mg_l,pH,specific_conductance_us_cm,temperature_c,turbidity_fnu,precipitation,rain,snowfall,snow_depth,soil_moisture_0_to_1cm,soil_moisture_1_to_3cm,temperature_2m,wind_speed_10m,vapour_pressure_deficit,precip_3hr,precip_24hr,precip_72hr
0,2010-07-06 00:00:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-07-06 00:15:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-07-06 00:30:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-07-06 00:45:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-07-06 01:00:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
626707,2026-07-06 23:00:00+00:00,NaN,NaN,6.6,8.6,356.0,34.9,6.0,1.0,1.0,0.0,0.0,NaN,NaN,24.65,0.648999,0.118415,29.8,50.2,74.800001
626708,2026-07-07 00:00:00+00:00,NaN,NaN,6.6,8.6,356.0,34.9,6.0,1.8,1.8,0.0,0.0,NaN,NaN,24.10,11.074022,0.062441,31.6,51.4,76.600000
626709,2026-07-07 01:00:00+00:00,NaN,NaN,6.6,8.6,356.0,34.9,6.0,0.7,0.7,0.0,0.0,NaN,NaN,23.45,11.225132,0.051792,32.3,51.5,77.300000
626710,2026-07-07 02:00:00+00:00,NaN,NaN,6.6,8.6,356.0,34.9,6.0,1.1,1.1,0.0,0.0,NaN,NaN,22.85,10.137692,0.016814,33.4,52.0,78.400000


In [5]:
# Flood Action Stage: 5 ft
# Minor Flood Stage: 10 ft
# Moderate Flood Stage: 12 ft
# Major Flood Stage: 14 ft
FLOOD_ACTION_STAGE = 5.0
MINOR_FLOOD_STAGE = 10.0
MODERATE_FLOOD_STAGE = 12.0
MAJOR_FLOOD_STAGE = 14.0

#df.hist(figsize=(10, 6))
# get the instances where Gage Height is > 5
df = df.dropna(subset=['gage_height_ft'])
df_flood = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE]
df_minor_flood = df[df['gage_height_ft'] > MINOR_FLOOD_STAGE]
df_moderate_flood = df[df['gage_height_ft'] > MODERATE_FLOOD_STAGE]
df_major_flood = df[df['gage_height_ft'] > MAJOR_FLOOD_STAGE]

# print the lengths of each of the dataframes
print(f"Total records: {len(df)}")
print(f"Flood Action Stage records: {len(df_flood)}")
print(f"Minor flood records: {len(df_minor_flood)}")
print(f"Moderate flood records: {len(df_moderate_flood)}")
print(f"Major flood records: {len(df_major_flood)}")

Total records: 624474
Flood Action Stage records: 66631
Minor flood records: 1199
Moderate flood records: 55
Major flood records: 0


In [6]:

# Name the 'Unnamed: 0' column as 'datetime' and convert it to datetime type
df['datetime'] = pd.to_datetime(df['Unnamed: 0'])

df = df.sort_values('datetime').reset_index(drop=True)

# how many hours of gap counts as "the storm ended" (tune this to your data's
# sampling frequency, e.g. 6-12h for hourly gauge data, 24-48h for daily)
GAP_HOURS = 12

# isolate just the flagged (action-stage) rows
flood_rows = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE].copy()

# time since previous flagged reading, saved in column 'gap'
flood_rows['gap'] = flood_rows['datetime'].diff()

# start a new event whenever the gap exceeds threshold (or it's the first row)
flood_rows['new_event'] = (
    flood_rows['gap'].isna() | (flood_rows['gap'] > pd.Timedelta(hours=GAP_HOURS))
)
flood_rows['event_id'] = flood_rows['new_event'].cumsum()

df = df.merge(
    flood_rows[['datetime', 'event_id']],
    on='datetime',
    how='left'
)

# summarize each event
events = flood_rows.groupby('event_id').agg(
    start=('datetime', 'min'),
    end=('datetime', 'max'),
    n_readings=('datetime', 'count'),
    peak_gage_height=('gage_height_ft', 'max')  # adjust column name as needed
).reset_index(drop=True)

events['duration_hours'] = (events['end'] - events['start']).dt.total_seconds() / 3600

print(f"Total flagged readings: {len(flood_rows)}")
print(f"Independent storm events: {len(events)}")
print(events)


C:\Users\drpri\AppData\Local\Temp\ipykernel_13240\2735280843.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['datetime'] = pd.to_datetime(df['Unnamed: 0'])


Total flagged readings: 66631
Independent storm events: 123
                        start                       end  n_readings  \
0   2010-12-03 00:00:00+00:00 2010-12-05 12:15:00+00:00         242   
1   2011-02-27 11:30:00+00:00 2011-03-04 14:45:00+00:00         494   
2   2011-03-07 01:30:00+00:00 2011-03-19 02:30:00+00:00        1153   
3   2011-03-26 06:30:00+00:00 2011-03-28 07:30:00+00:00         194   
4   2011-04-13 08:45:00+00:00 2011-05-08 04:15:00+00:00        2382   
..                        ...                       ...         ...   
118 2025-06-10 16:00:00+00:00 2025-06-13 06:00:00+00:00         193   
119 2025-06-16 22:15:00+00:00 2025-06-22 17:45:00+00:00         544   
120 2025-07-19 23:50:00+00:00 2025-07-20 01:35:00+00:00          22   
121 2026-02-21 15:35:00+00:00 2026-02-24 23:50:00+00:00         964   
122 2026-05-24 18:55:00+00:00 2026-05-31 21:10:00+00:00        2044   

     peak_gage_height  duration_hours  
0                6.70           60.25  
1      

In [7]:
LOOKAHEAD_HOURS = 24  # predict within next 24h
FREQ_MINUTES = 15     # Hydraulic data's sampling interval 

# Cqalculate how many rows ahead
lookahead_steps = int(LOOKAHEAD_HOURS * 60 / FREQ_MINUTES)

# 'will_flood' checks whether the flood stage will be exceeded in the next `lookahead_steps` readings, setting it as 
# 1 if any of the next readings exceed the flood action stage, and 0 otherwise. This is used as the target variable for the model.

df['will_flood'] = (
    df['gage_height_ft']
    .shift(-1)                                   
    .rolling(window=lookahead_steps, min_periods=1)
    .max()
    .shift(-(lookahead_steps - 1))                # align window to start right after current row
    > FLOOD_ACTION_STAGE
).astype(int)



In [8]:
import numpy as np

# make a column called gage_height_roc_1h and gage_height_roc_6h to see the rate of change in gage height over 1 hour and 6 hours, respectively 
df['gage_height_roc_1h'] = df['gage_height_ft'].diff(int(60/FREQ_MINUTES))
df['gage_height_roc_6h'] = df['gage_height_ft'].diff(int(360/FREQ_MINUTES))

df['gage_height_now'] = df['gage_height_ft']
df['streamflow_now'] = df['streamflow_cfs']

for col in ['precip_3hr', 'precip_24hr', 'precip_72hr', 'turbidity_fnu']:
    if col in df.columns:
        df[f'{col}_log'] = np.log1p(df[col].clip(lower=0))

feature_columns = [
    # hydraulic — current state + trend
    'gage_height_ft',
    'gage_height_roc_1h',
    'gage_height_roc_6h',

    # precip — log-transformed versions only (not the raw skewed ones)
    'precip_3hr_log',
    'precip_24hr_log',
    'precip_72hr_log',

    # weather
    'temperature_2m',
    'wind_speed_10m',
    'vapour_pressure_deficit',
    'rain',
    'snowfall',
    'snow_depth',

    # water quality
    'specific_conductance_us_cm',
    'temperature_c',
]


In [9]:
# tag every row (not just flood rows) with which event's "influence window" it falls in
# so the same storm doesn't appear in both train and test
events_sorted = events.sort_values('start').reset_index(drop=True)

# hold out the most recent ~20% of events as test
n_test_events = int(len(events_sorted) * 0.2)
test_events = events_sorted.iloc[-n_test_events:]
train_events = events_sorted.iloc[:-n_test_events]

test_start_cutoff = test_events['start'].min() - pd.Timedelta(days=3)  # buffer before first test storm

train_df = df[df['datetime'] < test_start_cutoff].dropna(subset=feature_columns + ['will_flood'])
test_df  = df[df['datetime'] >= test_start_cutoff].dropna(subset=feature_columns + ['will_flood'])

test_df

,Unnamed: 0,gage_height_ft,streamflow_cfs,dissolved_oxygen_mg_l,pH,specific_conductance_us_cm,temperature_c,turbidity_fnu,precipitation,rain,...,event_id,will_flood,gage_height_roc_1h,gage_height_roc_6h,gage_height_now,streamflow_now,precip_3hr_log,precip_24hr_log,precip_72hr_log,turbidity_fnu_log
430325,2022-12-14 13:30:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,-0.01,-0.03,3.13,3520.0,0.000000,0.832909,3.234749,0.875469
430326,2022-12-14 13:45:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,0.00,-0.03,3.13,3520.0,0.000000,0.788457,3.234749,0.875469
430327,2022-12-14 14:00:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,0.00,-0.03,3.13,3520.0,0.000000,0.741937,3.234749,0.875469
430328,2022-12-14 14:15:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,0.00,-0.03,3.13,3520.0,0.000000,0.693147,3.234749,0.875469
430329,2022-12-14 14:30:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,0.00,-0.02,3.13,3520.0,0.000000,0.641854,3.234749,0.875469
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
624469,2026-07-05 23:40:00+00:00,2.82,1980.0,6.6,8.6,356.0,34.9,6.0,0.0,0.0,...,NaN,0,-0.01,-0.01,2.82,1980.0,1.757858,3.063391,3.828641,1.945910
624470,2026-07-05 23:45:00+00:00,2.82,1980.0,6.6,8.6,356.0,34.9,6.0,0.0,0.0,...,NaN,0,-0.01,-0.01,2.82,1980.0,1.757858,3.063391,3.828641,1.945910
624471,2026-07-05 23:50:00+00:00,2.82,1980.0,6.6,8.6,356.0,34.9,6.0,0.0,0.0,...,NaN,0,0.00,-0.01,2.82,1980.0,1.757858,3.063391,3.828641,1.945910
624472,2026-07-05 23:55:00+00:00,2.82,1980.0,6.6,8.6,356.0,34.9,6.0,0.0,0.0,...,NaN,0,0.00,-0.01,2.82,1980.0,1.757858,3.063391,3.828641,1.945910


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


X_train, y_train = train_df[feature_columns], train_df['will_flood']
X_test, y_test = test_df[feature_columns], test_df['will_flood']

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
probs = model.predict_proba(X_test)[:, 1]

In [11]:
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    classification_report, roc_auc_score
)

probs = model.predict_proba(X_test)[:, 1]

# PR-AUC (Precision-Recall Area Under Curve) is used for imbalanced datasets, as it focuses on the performance
#  of the positive class (flood events) and is more informative than accuracy in such cases.
print(f"PR-AUC: {average_precision_score(y_test, probs):.3f}")

# ROC-AUC (Receiver Operating Characteristic Area Under Curve) is a performance measurement for classification 
# prediction problems at various threshold settings. It tells how much the model is capable of distinguishing
print(f"ROC-AUC: {roc_auc_score(y_test, probs):.3f}")

# don't default to 0.5 — pick a threshold that favors recall (missing a flood is worse)
precision, recall, thresholds = precision_recall_curve(y_test, probs)


PR-AUC: 0.971
ROC-AUC: 0.996


In [12]:
def lead_time_last_rise(event_rows, flood_start, threshold=0.5, min_below_hours=6, freq_minutes=15):
    """
    Find the most recent rise above threshold before the flood, requiring
    the probability to have dipped below threshold for at least
    `min_below_hours` beforehand — this separates a fresh, distinct rise
    from an unrelated earlier event or a brief blip.
    """
    trace = event_rows.sort_values('datetime').reset_index(drop=True)
    above = trace['predicted_prob'] >= threshold
    min_below_rows = int(min_below_hours * 60 / freq_minutes)

    crossings = trace.index[above & ~above.shift(1, fill_value=False)]

    valid_crossings = []
    for idx in crossings:
        if idx < min_below_rows:
            continue  # not enough prior history in this window to confirm a real dip — skip, don't assume
        lookback_start = idx - min_below_rows
        if not above.iloc[lookback_start:idx].any():
            valid_crossings.append(idx)

    if not valid_crossings:
        return None  # genuinely no valid crossing found — e.g. sustained risk the whole window

    last_idx = valid_crossings[-1]
    return trace.loc[last_idx, 'datetime']

In [13]:

test_df = test_df.copy()

# Add the predicted probabilities to the test dataframe for further analysis
test_df['predicted_prob'] = probs

ALERT_THRESHOLD = 0.716

# This finds all unique event IDs where the gage height exceeds the flood action stage, indicating a flood event.
flood_events = test_df.loc[test_df['gage_height_ft'] > FLOOD_ACTION_STAGE, 'event_id'].dropna().unique()

results = []

LOOKBACK_HOURS = 168  # how far before the flood to look for an early alert

results = []
for eid in flood_events:
    flood_rows_this_event = test_df[test_df['event_id'] == eid]
    flood_start = flood_rows_this_event['datetime'].min()
    window_start = flood_start - pd.Timedelta(hours=LOOKBACK_HOURS)
    event_rows = test_df[(test_df['datetime'] >= window_start) & (test_df['datetime'] <= flood_rows_this_event['datetime'].max())]
    alert_time = lead_time_last_rise(event_rows, flood_start)
    lead_time = (flood_start - alert_time).total_seconds() / 3600 if alert_time is not None else None

    results.append({'event_id': eid, 'lead_time_hours': lead_time, 'peak_prob': flood_rows_this_event['predicted_prob'].max()})


results_df = pd.DataFrame(results)
print(results_df)
print(f"\nMedian lead time: {results_df['lead_time_hours'].median():.1f}h")

    event_id  lead_time_hours  peak_prob
0      100.0         4.250000   0.991608
1      101.0         6.000000   0.996646
2      102.0        20.000000   1.000000
3      103.0         4.000000   1.000000
4      104.0         8.250000   0.987194
5      105.0         9.000000   1.000000
6      106.0         6.000000   1.000000
7      107.0        20.750000   0.999965
8      108.0        12.250000   0.997788
9      109.0        13.250000   1.000000
10     110.0         9.500000   1.000000
11     111.0         5.750000   1.000000
12     112.0         0.000000   0.992853
13     113.0         4.000000   1.000000
14     114.0         7.750000   1.000000
15     115.0         4.000000   1.000000
16     116.0         7.000000   0.982493
17     117.0        21.000000   1.000000
18     118.0        29.250000   1.000000
19     119.0         9.000000   0.996560
20     120.0        19.750000   1.000000
21     121.0         0.416667   0.738804
22     122.0         4.083333   0.960419
23     123.0    

In [14]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, probs)

# find threshold that gives recall >= some target, e.g. 0.90
import numpy as np
target_recall = 0.90
qualifying = np.where(recall >= target_recall)[0]
idx = qualifying[-1] if len(qualifying) > 0 else -1  # take the LAST (highest-threshold) index, not the first

print(f"Threshold for {target_recall} recall: {thresholds[idx]:.3f}, precision at that point: {precision[idx]:.3f}")

Threshold for 0.9 recall: 0.411, precision at that point: 0.922


In [15]:
from sklearn.metrics import confusion_matrix
final_threshold = thresholds[idx]
preds = (probs >= final_threshold).astype(int)
print(confusion_matrix(y_test, preds))

[[180067    995]
 [  1308  11779]]


In [16]:
# Generate a classification report
print(classification_report(y_test, preds, target_names=['No Flood', 'Flood']))

              precision    recall  f1-score   support

    No Flood       0.99      0.99      0.99    181062
       Flood       0.92      0.90      0.91     13087

    accuracy                           0.99    194149
   macro avg       0.96      0.95      0.95    194149
weighted avg       0.99      0.99      0.99    194149

